# Multi-Query

표현이 다른 여러 질문의 후보를 합치면 두 근거를 함께 찾는지 확인합니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** Multi-Query는 원 질문에서 의미가 비슷한 검색 질문 여러 개를 만들고 각 질문의 후보를 합칩니다. 이 절의 구현은 문서 ID가 같은 후보에서는 가장 높은 코사인 유사도를 남기고 그 점수로 내림차순 정렬합니다. RRF는 이 절에서 구현하지 않으며 09절의 패턴 비교에서 사용합니다.

**다른 상황에 적용.** ‘육아휴직 급여’라는 질문을 지급 조건, 신청 절차, 기간의 관점으로 바꾸면 서로 다른 문서에 흩어진 근거를 모을 수 있습니다. 관점이 달라도 원 질문의 대상과 제한 조건은 유지해야 합니다.

**비교와 경계.** Multi-Query는 표현 차이를 넓히지만 하나의 질문에 여러 사실이 섞인 문제를 분리하지는 않습니다. 각 하위 질문에 따로 답해야 한다면 질의 분해가 더 알맞습니다. 후보를 늘렸다는 사실만으로 Recall이 향상됐다고 말할 수 없으며 정답 문서 ID로 측정해야 합니다.


## 실행 환경 준비

고정한 Python 패키지를 확인합니다.

**이 셀의 코드 읽기.** `%pip install`은 이 노트북 커널이 사용할 패키지를 설치합니다. 앞 셀의 값을 입력으로 사용하지 않으며, 설치가 끝난 뒤의 코드 셀에서 패키지를 가져와 사용합니다.


In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 numpy==2.5.3
import sys
print("준비 완료:", sys.version.split()[0])

Note: you may need to restart the kernel to use updated packages.
준비 완료: 3.12.13


## .env 찾기

가까운 .env에서 키를 읽고 존재 여부만 표시합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [2]:
# 현재 폴더에서 가까운 .env를 찾되 비밀 값은 출력하지 않습니다.
import os, time
from dotenv import find_dotenv, load_dotenv
env_path=find_dotenv(usecwd=True)
load_dotenv(env_path)
print("OPENAI_API_KEY:","있음" if os.getenv("OPENAI_API_KEY") else "없음")

OPENAI_API_KEY: 있음


## 공통 데이터

Ch12의 축소 데이터와 Ch11 방식의 정답 문서 ID를 사용합니다.

**이 셀의 코드 읽기.** `json.loads(text)`는 JSON 문자열을 파이썬 딕셔너리나 목록으로 변환해 반환합니다. `read_text()`는 파일 내용을 문자열로 반환합니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
# 배포 묶음에서도 같은 상대 경로를 찾도록 프로젝트 루트를 거슬러 올라갑니다.
import json
from pathlib import Path
import numpy as np
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"labs").is_dir())
data_dir=root/"labs/part4/ch14-rag-patterns/data"
docs=json.loads((data_dir/"documents.json").read_text(encoding="utf-8"))
queries=json.loads((data_dir/"queries.json").read_text(encoding="utf-8"))
print("문서·질문:",len(docs),len(queries))

문서·질문: 12 8


## 모델과 임베딩

모델과 256차원 문서 벡터를 준비합니다.

**이 셀의 코드 읽기.** `OpenAIEmbeddings(...)`는 문자열을 수치 벡터로 변환할 임베딩 클라이언트를 준비합니다. 실제 호출은 `embed_query()`나 `embed_documents()`에서 일어납니다. `embed_documents(texts)`는 여러 문장을 입력받아 문장 수만큼의 벡터를 반환합니다. `init_chat_model(...)`은 채팅 모델 객체를 만들며, 이 줄에서는 아직 답변을 생성하지 않습니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [4]:
# 256차원은 Ch12와 같은 비교 조건이며 행 하나가 문서 하나입니다.
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
chat=init_chat_model("gpt-4.1-mini",model_provider="openai",temperature=0)
embedder=OpenAIEmbeddings(model="text-embedding-3-small",dimensions=256)
vectors=np.array(embedder.embed_documents([d["text"] for d in docs]))
print("임베딩 배열:",vectors.shape)

임베딩 배열: (12, 256)


## 검색·생성 함수

검색 점수, 토큰, 시간, 추정 비용을 같은 방식으로 기록합니다.

**이 셀의 코드 읽기.** `search(question: str, k: int = 3)`, `ask(prompt: str)`, `show_hits(hits: list[tuple[dict[str, Any], float]])`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `invoke(input)`는 정의한 함수를 뒤에서 호출할 때 해당 함수 본문 안에서 모델이나 그래프를 실행하고 응답 객체를 반환합니다. `embed_query(text)`는 검색 질문 한 개를 벡터 하나로 반환합니다.


In [5]:
# 코사인 유사도는 내적을 두 벡터의 길이로 나누어 계산합니다.
from typing import Any
usage_log: list[dict[str, float | int]] = []

def search(question: str, k: int = 3) -> list[tuple[dict[str, Any], float]]:
    q=np.array(embedder.embed_query(question))
    scores=vectors@q/(np.linalg.norm(vectors,axis=1)*np.linalg.norm(q))
    # argsort 결과를 뒤집어 점수가 큰 문서부터 k개를 고릅니다.
    return [(docs[i],float(scores[i])) for i in np.argsort(scores)[::-1][:k]]
def ask(prompt: str) -> tuple[str, int, float]:
    started=time.perf_counter()
    msg=chat.invoke(prompt)
    usage=msg.usage_metadata or {}
    details=usage.get("input_token_details", {})
    input_tokens=usage.get("input_tokens", 0)
    cached_tokens=details.get("cache_read", 0)
    output_tokens=usage.get("output_tokens", 0)
    tokens=usage.get("total_tokens", 0)
    # 2026-09-10 gpt-4.1-mini 공개 단가로 추정하며 실제 청구액과 다를 수 있습니다.
    usd=((input_tokens-cached_tokens)*0.40+cached_tokens*0.10+output_tokens*1.60)/1_000_000
    usage_log.append({"input":input_tokens,"cached":cached_tokens,"output":output_tokens,"usd":usd})
    return msg.text,tokens,round((time.perf_counter()-started)*1000,1)

def show_hits(hits: list[tuple[dict[str, Any], float]]) -> list[tuple[str, float]]:
    return [(d["doc_id"],round(score,3)) for d,score in hits]

## 기준선 검색

원 질문 하나로 상위 두 문서를 검색합니다. 두 정답 문서 가운데 빠진 문서가 있는지 확인합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [6]:
# 원문 질문 하나로 검색한 결과를 비교 기준으로 저장합니다.
question="책을 빌리는 기간과 늦게 반납했을 때 조치를 알려 주세요."
baseline=search(question,k=2)
print("기준선:",show_hits(baseline))

기준선: [('LIB-001', 0.466), ('LIB-004', 0.443)]


## 세 검색 질문 생성

원 질문을 세 가지 검색 표현으로 바꿉니다. 출력 질문이 원래 대상과 조건을 유지하는지 확인합니다.

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [7]:
# 같은 의도를 다른 표현으로 바꾼 검색 질문을 최대 세 개 만듭니다.
text,tokens,latency=ask(f"검색 관점을 달리한 한국어 질문 3개를 줄마다 하나씩 쓰세요. 원문:{question}")
alternatives=[x.lstrip("-123. ") for x in text.splitlines() if x.strip()][:3]
print(alternatives,"토큰:",tokens,"ms:",latency)

['도서 대출 기간은 얼마나 되나요?  ', '책을 제때 반납하지 않으면 어떤 벌칙이 있나요?  ', '연체 시 처리 절차와 벌금 규정이 어떻게 되나요?'] 토큰: 100 ms: 1114.3


## 후보 합치기

원 질문과 변형 질문의 결과를 문서 ID로 합칩니다. 병합 후보와 두 정답 문서의 포함 여부를 확인합니다.

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [8]:
# 문서 ID로 중복을 제거하고 문서별 최고 유사도를 보존합니다.
pool={}
for q in [question,*alternatives]:
    for doc,score in search(q,k=2):
        pool[doc["doc_id"]]=(doc,max(score,pool.get(doc["doc_id"],(doc,-1))[1]))
merged=sorted(pool.values(),key=lambda x:x[1],reverse=True)
print("통합 후보:",show_hits(merged))
print("정답 포함:",{"LIB-001","LIB-002"}.issubset(pool))

통합 후보: [('LIB-002', 0.492), ('LIB-001', 0.466), ('LIB-003', 0.445), ('LIB-004', 0.443), ('FAC-004', 0.335)]
정답 포함: True
